# Решения: перебор и сдача

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

import itertools
import matplotlib.pyplot as plt
from pathlib import Path as _P
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

_P('figures').mkdir(exist_ok=True)
X_rest, X_final, y_rest, y_final = train_test_split(
    df[PIXELS], df['label'], test_size=0.2, random_state=0, stratify=df['label'])
X_fit, X_val, y_fit, y_val = train_test_split(
    X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)


## Урок. 1–3. Сетка настроек и подмножества признаков

In [ ]:
K_VALUES = [1, 3, 5, 7, 9]
SCALING = [False, True]
grid = list(itertools.product(K_VALUES, SCALING))


def scale_by_train(fit_frame, other_frame):
    mins = fit_frame.min()
    rng = (fit_frame.max() - fit_frame.min()).replace(0, 1)
    return (fit_frame - mins) / rng, (other_frame - mins) / rng


rows = []
for k, scaled in grid:
    a, b = scale_by_train(X_fit, X_val) if scaled else (X_fit, X_val)
    m = KNeighborsClassifier(n_neighbors=k).fit(a, y_fit)
    rows.append([k, scaled, float(accuracy_score(y_val, m.predict(b)))])
exp_table = pd.DataFrame(rows, columns=['k', 'scaled', 'accuracy'])


def make_features(frame):
    out = pd.DataFrame(index=frame.index)
    out['ink'] = frame.sum(axis=1)
    out['n_dark'] = (frame > 8).sum(axis=1)
    out['top_ink'] = frame[PIXELS[:32]].sum(axis=1)
    out['bottom_ink'] = frame[PIXELS[32:]].sum(axis=1)
    return out


FEATURES = ['ink', 'n_dark', 'top_ink', 'bottom_ink']
feat_fit, feat_val = make_features(X_fit), make_features(X_val)
combo_rows = []
for pair in itertools.combinations(FEATURES, 2):
    m = KNeighborsClassifier(5).fit(feat_fit[list(pair)], y_fit)
    combo_rows.append(['+'.join(pair), float(accuracy_score(y_val, m.predict(feat_val[list(pair)])))])
combo_table = pd.DataFrame(combo_rows, columns=['features', 'accuracy'])
print(exp_table)
print(combo_table)

## Урок. 4–6. Выбор, честная оценка, файлы

In [ ]:
best = exp_table.sort_values('accuracy', ascending=False).iloc[0]
best_k, best_scaled = int(best['k']), bool(best['scaled'])
a, b = scale_by_train(X_fit, X_final) if best_scaled else (X_fit, X_final)
final_model = KNeighborsClassifier(n_neighbors=best_k).fit(a, y_fit)
acc_final = float(accuracy_score(y_final, final_model.predict(b)))
baseline_final = float((y_final == y_fit.value_counts().idxmax()).mean())
peek = []
for k, scaled in grid:
    aa, bb = scale_by_train(X_fit, X_final) if scaled else (X_fit, X_final)
    m = KNeighborsClassifier(n_neighbors=k).fit(aa, y_fit)
    peek.append(float(accuracy_score(y_final, m.predict(bb))))
acc_peek = max(peek)
optimism = acc_peek - acc_final
OPTIMISM_NOTE = (
    'Максимум по десяти настройкам на финальной части — уже результат подбора: '
    'мы выбрали то, что случайно лучше подошло к этим 360 картинкам. '
    'Отчётное число должно приходить от настройки, выбранной без них.'
)
csv_path = _P('experiments.csv')
exp_table.to_csv(csv_path, index=False)
plt.figure()
for scaled in (False, True):
    part = exp_table[exp_table['scaled'] == scaled]
    plt.plot(part['k'], part['accuracy'], marker='o', label=f'scaled={scaled}')
plt.xlabel('k (число соседей)'); plt.ylabel('точность на проверочной части')
plt.legend(); plt.tight_layout()
plot_path = _P('figures/config_accuracy.png'); plt.savefig(plot_path); plt.close()
print(best_k, best_scaled, round(acc_final, 4), round(baseline_final, 4),
      round(acc_peek, 4), round(optimism, 4))

## Урок. 7–9. Чек-лист, отчёт, мост к модулю 5

In [ ]:
acceptance = pd.Series(
    [True, True, True, True, True, True],
    index=['baseline', 'protocol', 'experiments_csv', 'figure', 'final_score', 'limitations'],
)
n_final_looks = 1
REPORT = (
    f'Данные: 1797 картинок цифр 8x8, значения пикселя 0..16. Протокол: 60/20/20 со stratify, '
    f'настройки выбирались по проверочной части, финальная часть использована один раз. '
    f'Выбрано k={best_k}, масштабирование={best_scaled}. Точность на финальной части '
    f'{acc_final:.3f} против baseline самой частой цифры {baseline_final:.3f}. '
    f'Ограничения: картинки 8x8 вместо полного разрешения, kNN хранит всю обучающую часть '
    f'и отвечает тем дольше, чем больше данных; поток реальной почты может быть смещён.'
)
READY = True
NEXT_MODULE = (
    'kNN не обучается заранее: на каждый конверт он сравнивает картинку со всей обучающей частью, '
    'поэтому миллион примеров означает миллион сравнений на один ответ. '
    'Дальше учимся строить признаки — они дают больше, чем замена модели.'
)
print(acceptance.all(), len(REPORT), READY)

## ДЗ. 1–4

In [ ]:
rows = []
for k, metric in itertools.product([1, 3, 5, 7, 9], ['euclidean', 'manhattan']):
    m = KNeighborsClassifier(n_neighbors=k, metric=metric).fit(X_fit, y_fit)
    rows.append([k, metric, float(accuracy_score(y_val, m.predict(X_val)))])
metric_table = pd.DataFrame(rows, columns=['k', 'metric', 'accuracy'])
best_row = metric_table.sort_values('accuracy', ascending=False).iloc[0]
csv_path = _P('metric_experiments.csv')
metric_table.to_csv(csv_path, index=False)
report_md = (
    '## Данные\n1797 картинок 8x8, десять цифр, доли классов почти равные.\n\n'
    '## Протокол\n60/20/20 со stratify; настройки выбираются по проверочной части; '
    'финальная часть — один раз.\n\n'
    f'## Опыты\nПеребрано {len(metric_table)} настроек (k x способ измерять расстояние). '
    f"Лучшая: k={int(best_row['k'])}, {best_row['metric']}, "
    f"точность на проверочной {best_row['accuracy']:.3f}.\n\n"
    '## Итог\nРаспознаватель уверенно бьёт baseline самой частой цифры (~0.10).\n\n'
    '## Ограничения\nМелкие картинки 8x8; kNN медленный на больших данных; '
    'реальный поток индексов может быть смещён по цифрам.'
)
REFLECTION = (
    'В начале модуля точность казалась одним числом про модель. Теперь видно, что число зависит '
    'от разбиения, от baseline, от масштаба признаков и от того, сколько раз мы подглядывали '
    'в проверочные данные. Честный результат — это протокол, а не удачный запуск.'
)
HONEST_LOOKS = 1
print(metric_table)
print(best_row)